In [2]:
import os
import numpy as np

data_path = "C:/Users/USER/Downloads/760-FL-Parkinson-MRI-main/preprocessed_data"
output_base = "C:/Users/USER/Downloads/760-FL-Parkinson-MRI-main/client_data"

# Step 1: Load data with filenames
X = []
filenames = []

for file in os.listdir(data_path):
    if file.endswith(".npy"):
        vol = np.load(os.path.join(data_path, file))
        X.append(vol)
        filenames.append(file)

X = np.array(X)
num_total = len(X)

# Step 2: Shuffle indices
indices = np.arange(num_total)
np.random.shuffle(indices)

# Step 3: Split indices
split1 = int(0.44 * num_total)

client_indices_list = [
    indices[:split1],       # Client 1: 44%
    indices[split1:],       # Client 3: 50%
]

# Step 4: Save each client's data in a separate folder
for i, client_indices in enumerate(client_indices_list):
    client_folder = os.path.join(output_base, f"client_{i+1}")
    os.makedirs(client_folder, exist_ok=True)

    for idx in client_indices:
        filename = filenames[idx]
        filepath = os.path.join(client_folder, filename)
        np.save(filepath, X[idx])

print("Client datasets saved with original filenames.")


Client datasets saved with original filenames.


In [2]:
def extract_label_from_filename(filename):
    # Example filename: volume_01_label_0_stripped.npy
    try:
        parts = filename.split('_')
        label_index = parts.index("label") + 1
        return int(parts[label_index])
    except (ValueError, IndexError):
        raise ValueError(f"Cannot extract label from filename: {filename}")


In [5]:
import os
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split

# --- Settings ---
data_path = "C:/Users/USER/Downloads/760-FL-Parkinson-MRI-main/preprocessed_data"
output_base = "C:/Users/USER/Downloads/760-FL-Parkinson-MRI-main/client_data1"
num_clients = 2
alpha = 0.5  # Dirichlet concentration parameter (lower = more non-IID)

# --- Load data and labels ---
X = []
y = []          # Labels must be provided or extracted somehow
filenames = []

for file in os.listdir(data_path):
    if file.endswith(".npy"):
        vol = np.load(os.path.join(data_path, file))
        label = extract_label_from_filename(file)  # You must define this
        X.append(vol)
        y.append(label)
        filenames.append(file)

X = np.array(X)
y = np.array(y)
filenames = np.array(filenames)

# --- Group indices by class ---
class_indices = defaultdict(list)
for idx, label in enumerate(y):
    class_indices[label].append(idx)

# --- Dirichlet distribution split ---
client_indices = [[] for _ in range(num_clients)]

for cls, indices in class_indices.items():
    np.random.shuffle(indices)
    proportions = np.random.dirichlet([alpha] * num_clients)
    proportions = (np.cumsum(proportions) * len(indices)).astype(int)[:-1]
    splits = np.split(indices, proportions)

    for i, split in enumerate(splits):
        client_indices[i].extend(split)

# --- Save client datasets ---
for i, indices in enumerate(client_indices):
    client_folder = os.path.join(output_base, f"client_{i+1}")
    os.makedirs(client_folder, exist_ok=True)
    
    for idx in indices:
        filename = filenames[idx]
        filepath = os.path.join(client_folder, filename)
        np.save(filepath, X[idx])

print("Client datasets saved with Dirichlet-distributed class balance.")


Client datasets saved with Dirichlet-distributed class balance.
